Import Libraries

In [1]:
import pandas as pd
import  matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load the dataset from the CSV file sourced from Kaggle 
raw = pd.read_csv('../data/fraud_detection.csv')
# Drop the 'transaction_id' column from the raw dataframe, because it is not needed for analysis
raw = raw.drop('transaction_id', axis=1)
# Display the first 10 rows of the dataframe
raw.head(10)

In [ ]:
raw.info()

In [ ]:
raw['merchant_type'].value_counts()

In [ ]:
raw['device_type'].value_counts()

There are 3 indenpendent variables which are 
1. amount: Shows the amount of one transaction
2. merchant_type: The type of product/service a transaction is in
3. device_type: The type of product/service a transaction is in

Visualize each independent variable relation to the target variable

In [ ]:
# Boxplot: Amount vs Label
plt.figure(figsize=(8, 5))
sns.boxplot(x='label', y='amount', data=raw)
plt.title('Transaction Amount by Label')
plt.show()


Those which are fraudulent and not have similar median, mean, and quartiles. But those which are not fraudulent, have multiple transactions that tend to be higher in value. 

In [ ]:
# Countplot: Merchant Type vs Label
plt.figure(figsize=(10, 5))
sns.countplot(x='merchant_type', hue='label', data=raw)
plt.title('Merchant Type by Label')
plt.xticks(rotation=45)
plt.show()

There are no significant conclusion to be drawn from the relation between merchant_type and label.

In [ ]:
# Countplot: Device Type vs Label
plt.figure(figsize=(8, 5))
sns.countplot(x='device_type', hue='label', data=raw)
plt.title('Device Type by Label')
plt.xticks(rotation=45)
plt.show()

There are no significant conclusion to be drawn from the relation between merchant_type and label.


In [ ]:
# Grouped barplot to visualize merchant_type, device_type, and label in one graph
plt.figure(figsize=(12, 6))
sns.countplot(
    x='merchant_type',
    hue='label',
    data=raw,
    palette='Set2'
)
plt.title('Merchant Type by Label')
plt.xticks(rotation=45)
plt.legend(title='Label')
plt.tight_layout()
plt.show()

# FacetGrid to show device_type distribution within each merchant_type and label
g = sns.catplot(
    x='device_type',
    hue='label',
    col='merchant_type',
    data=raw,
    kind='count',
    col_wrap=3,
    height=4,
    palette='Set2'
)
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Device Type and Label Distribution by Merchant')

In [ ]:
one_hot_data = pd.get_dummies(raw, columns=['merchant_type', 'device_type'], dtype='int')
# Display the first 10 rows of the one-hot encoded dataframe
one_hot_data.head(10)

In [ ]:
one_hot_data.corr()[['label']]

**Preprocessing Data**
Prepare data from raw dataframe and list the categorical variables for Sklearn Pipeline.

In [ ]:
from imblearn.over_sampling import SMOTENC

In [ ]:
categorial_features = ['merchant_type', 'device_type']
numerical_features = ['amount']
independent_var = categorial_features + numerical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    raw[independent_var], raw['label'], 
    test_size=0.2, random_state=42, stratify = raw['label'])

In [ ]:
sm = SMOTENC(categorical_encoder=categorical_features, random_state=42)

In [ ]:
X_res, y_res = sm.fit_resample(X, y)

**Model: Hyperparameter Tuning**

Set Tracking to MLFlow uri

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://mlflow.team-24.com")
mlflow.set_experiment("fraud_detection_experiment")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.base import clone
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn import set_config
from sklearn.model_selection import StratifiedKFold

In [ ]:
base_logreg = LogisticRegression(random_state=42, multi_class='ovr', penalty=None)
base_rf = RandomForestClassifier(random_state=42)
base_svc = SVC(random_state=42)

skf = StratifiedKFold(shuffle = False, n_splits=5)

Create parameter grids for each model type

In [ ]:
param_logreg = {'Logistic Regression__solver': ['lbfgs','newton-cg','sag']}
param_rf = {'n_estimators': [50, 100, 250, 500]
            , 'min_samples_leaf': [1, 2, 5]
            , 'min_samples_split': [10, 25, 50, 100]
}
param_svc = {'kernel': ['linear', 'rbf', 'poly']}

Create a pipeline for each type of Machine Learning algorithm.
Generally, logistic regression and SVM require normalization.
While tree based models are less sensitive to feature scales.

In [ ]:
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), numerical_features),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorial_features)
])

pipeline_logreg = Pipeline([
    ('Preprocessing_Column', preprocessor),
    ('Logistic Regression', base_logreg)
])

pipeline_svc = Pipeline([
    preprocessor,
    base_svc
])

Start tracking model in MLFlow.

In [ ]:
with mlflow.start_run(run_name="Logistic Regression Hyperparameter Tuning"):
    # Create and fit GridSearchCV
    gs_logreg = GridSearchCV(
        pipeline_logreg, param_logreg
        , cv=skf, scoring="precision"
    )

    gs_logreg.fit(X_train, y_train)

    # Best model evaluation
    best_score = gs_logreg.score(X_test, y_test)

    mlflow.log_param("model", "logistic_regression")
    mlflow.log_metric("test_score", best_score)
    print(f"Best parameters: {gs_logreg.best_params_}")
    print(f"Best cross-validation score: {gs_logreg.best_score_:.3f}")
    print(f"Test score: {best_score:.3f}")

In [ ]:
logreg_model1 = gs_logreg.best_estimator_

In [ ]:
logreg_model1.fit(X_train, y_train)

In [ ]:
y_pred_logreg1 = logreg_model1.predict(X_test)

In [ ]:
y_pred_logreg1

In [ ]:
X_test

In [ ]:
y_test.value_counts()